In [2]:
import pandas as pd
df = pd.read_csv('q3_retail_promotions.csv')

df['transaction_date'] = pd.to_datetime(df['transaction_date'])

df['year'] = df['transaction_date'].dt.year
df['month'] = df['transaction_date'].dt.month
df['day_of_week'] = df['transaction_date'].dt.dayofweek
df['is_month_end'] = (df['transaction_date'].dt.day >= 25).astype(int)

df = df.sort_values('transaction_date')

split = int(len(df)*0.8)
train = df[:split]
test = df[split:]

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression

cat = ['promotion_type','location_type','store_size']
num = [c for c in df.columns if c not in cat+['items_sold','transaction_date']]

pre = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat),
    ('num', StandardScaler(), num)
])

from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

for model in [LinearRegression(), RandomForestRegressor(random_state=42)]:
    pipe = Pipeline([('prep', pre), ('model', model)])
    pipe.fit(train, train['items_sold'])
    preds = pipe.predict(test)
    print(np.sqrt(mean_squared_error(test['items_sold'], preds)))


27.121451164890626
30.841610008504205


Pipeline ensures consistent preprocessing and modelling.

Time-based split is used to avoid data leakage.

ate features are extracted to capture time patterns.